# Tarea 03: Generacion de imagenes con autoencoder: Pytorch Lightning - Hydra - WandB




In [19]:

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
!pip -q install pytorch-lightning hydra-core wandb torchmetrics scikit-learn pandas


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## Dataset y DataModule

In [17]:
import os, random
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
import pytorch_lightning as pl
from PIL import Image
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from torchvision.utils import make_grid
from torchmetrics.image import StructuralSimilarityIndexMeasure
from sklearn.manifold import TSNE
from omegaconf import OmegaConf
from hydra import initialize_config_dir, compose
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import ModelCheckpoint


class MVTecDataset(Dataset):
    # Devuelve (imagen, label 0=buena/1=defectuosa, id_de_clase).
    # samples guarda (ruta, label, tipo_de_defecto, id_de_clase) para los histogramas.
    def __init__(self, root, classes, split, img_size=128):
        self.tf = transforms.Compose([transforms.Resize((img_size, img_size)), transforms.ToTensor()])
        self.samples = []
        for ci, c in enumerate(classes):
            base = Path(root) / c / split
            for sub in sorted(p for p in base.iterdir() if p.is_dir()):
                label = 0 if sub.name == "good" else 1
                for f in sorted(sub.glob("*.png")):
                    self.samples.append((str(f), label, sub.name, ci))

    def __len__(self): return len(self.samples)

    def __getitem__(self, i):
        path, label, _, ci = self.samples[i]
        img = Image.open(path).convert("RGB")
        return self.tf(img), label, ci


class MVTecDataModule(pl.LightningDataModule):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg.data

    def setup(self, stage=None):
        d = self.cfg
        full = MVTecDataset(d.root, d.classes, "train", d.img_size)
        n_val = int(len(full) * d.val_split)
        g = torch.Generator().manual_seed(0)
        self.train_ds, self.val_ds = random_split(full, [len(full) - n_val, n_val], generator=g)
        self.test_ds = MVTecDataset(d.root, d.classes, "test", d.img_size)

    def train_dataloader(self):
        return DataLoader(self.train_ds, self.cfg.batch_size, shuffle=True, num_workers=self.cfg.num_workers)
    def val_dataloader(self):
        return DataLoader(self.val_ds, self.cfg.batch_size, num_workers=self.cfg.num_workers)
    def test_dataloader(self):   # sin shuffle: el orden coincide con test_ds.samples
        return DataLoader(self.test_ds, self.cfg.batch_size, shuffle=False, num_workers=self.cfg.num_workers)

## Modelo: Variational Autoencoder

In [18]:
def enc_block(i, o): return nn.Sequential(nn.Conv2d(i, o, 4, 2, 1), nn.BatchNorm2d(o), nn.LeakyReLU(0.2))
def dec_block(i, o): return nn.Sequential(nn.ConvTranspose2d(i, o, 4, 2, 1), nn.BatchNorm2d(o), nn.ReLU())

class VAE(nn.Module):
    def __init__(self, z_dim=128, c=32):
        super().__init__()
        ch = [c, c*2, c*4, c*8, c*16]                       # 32,64,128,256,512
        self.encoder = nn.Sequential(enc_block(3, ch[0]), *[enc_block(ch[i], ch[i+1]) for i in range(4)])  # 128 -> 4
        self.flat = ch[-1] * 4 * 4
        self.fc_mu, self.fc_logvar = nn.Linear(self.flat, z_dim), nn.Linear(self.flat, z_dim)
        self.fc_dec = nn.Linear(z_dim, self.flat)
        self.ch_last = ch[-1]
        self.decoder = nn.Sequential(
            dec_block(ch[4], ch[3]), dec_block(ch[3], ch[2]), dec_block(ch[2], ch[1]), dec_block(ch[1], ch[0]),
            nn.ConvTranspose2d(ch[0], 3, 4, 2, 1), nn.Sigmoid())                                            # 4 -> 128

    def encode(self, x):
        h = self.encoder(x).flatten(1)
        return self.fc_mu(h), self.fc_logvar(h)

    def decode(self, z):
        return self.decoder(self.fc_dec(z).view(-1, self.ch_last, 4, 4))

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = mu + torch.randn_like(mu) * torch.exp(0.5 * logvar) if self.training else mu
        return self.decode(z), mu, logvar


def recon_loss(kind, x_hat, x, ssim_fn):
    if kind == "l1": return F.l1_loss(x_hat, x)
    if kind == "l2": return F.mse_loss(x_hat, x)
    if kind == "ssim": return 1 - ssim_fn(x_hat, x)
    if kind == "ssim_l1": return (1 - ssim_fn(x_hat, x)) + F.l1_loss(x_hat, x)
    raise ValueError(kind)


class LitVAE(pl.LightningModule):
    def __init__(self, z_dim, base_channels, loss, beta, lr, **kwargs):
        super().__init__()
        self.save_hyperparameters()
        self.net = VAE(z_dim, base_channels)
        self.ssim = StructuralSimilarityIndexMeasure(data_range=1.0)
        self.test_err = []

    def forward(self, x): return self.net(x)

    def _step(self, batch, stage):
        x, _, _ = batch
        x_hat, mu, logvar = self(x)
        rec = recon_loss(self.hparams.loss, x_hat, x, self.ssim)
        kl = -0.5 * torch.mean(torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1))
        loss = rec + self.hparams.beta * kl
        self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.log(f"{stage}_rec", rec, on_step=False, on_epoch=True)
        self.log(f"{stage}_kl", kl, on_step=False, on_epoch=True)
        with torch.no_grad():   # métricas para comparar entre funciones de pérdida
            self.log(f"{stage}_mse", F.mse_loss(x_hat, x), on_step=False, on_epoch=True)
            self.log(f"{stage}_ssim", self.ssim(x_hat, x), on_step=False, on_epoch=True)
        return loss

    def training_step(self, batch, i): return self._step(batch, "train")
    def validation_step(self, batch, i): self._step(batch, "val")

    def on_test_epoch_start(self): self.test_err = []
    def test_step(self, batch, i):
        x, _, _ = batch
        x_hat, _, _ = self(x)
        self.test_err.append(((x_hat - x) ** 2).mean(dim=(1, 2, 3)).cpu())   # error por imagen

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)